### Environment management tools
Because 
 is not a deployment strategy.

---
## Part 1 — Inspect the current environment
Before reaching for any tool, it's worth understanding what Python you're actually running inside this notebook.

In [ ]:
import sys
import os

print(f"Python version : {sys.version}")
print(f"Executable     : {sys.executable}")
print(f"Prefix         : {sys.prefix}")
print(f"VIRTUAL_ENV    : {os.environ.get('VIRTUAL_ENV', '(not set — are you in a venv?')}")

In [ ]:
# See every package currently installed in this environment
!pip list --format=columns

In [ ]:
# pip freeze gives you version-pinned output — the starting point for a requirements.txt
# Notice: ALL transitive deps appear here, not just the ones you explicitly installed.
!pip freeze | head -30

---
## Part 2 — `venv`: the built-in baseline

`venv` ships with Python and gives you a lightweight isolated package store.  
It does **not** manage which Python version you use — that's your responsibility.

Typical workflow:
```bash
python3 -m venv .venv          # create
source .venv/bin/activate       # activate (Linux/Mac)
pip install -r requirements.txt # install
deactivate                      # leave
```

We'll create a throwaway venv in `/tmp` so we don't pollute this project.

In [ ]:
import subprocess, shutil, os

DEMO_DIR = "/tmp/env_tools_demo"
shutil.rmtree(DEMO_DIR, ignore_errors=True)
os.makedirs(DEMO_DIR)

# Create the venv
result = subprocess.run(
    [sys.executable, "-m", "venv", f"{DEMO_DIR}/venv_demo"],
    capture_output=True, text=True
)
print(result.stdout or "venv created successfully")
print(result.stderr or "")

# Show what got created
!ls -1 /tmp/env_tools_demo/venv_demo/

In [ ]:
# The venv has its own Python binary that points back into the isolated site-packages
venv_python = f"{DEMO_DIR}/venv_demo/bin/python"

result = subprocess.run(
    [venv_python, "-c", "import sys; print(sys.executable); print(sys.prefix)"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Fresh venv: almost nothing installed
result = subprocess.run(
    [f"{DEMO_DIR}/venv_demo/bin/pip", "list", "--format=columns"],
    capture_output=True, text=True
)
print(result.stdout)

In [ ]:
# Install something into it and freeze
subprocess.run(
    [f"{DEMO_DIR}/venv_demo/bin/pip", "install", "requests", "--quiet"],
    capture_output=True
)

result = subprocess.run(
    [f"{DEMO_DIR}/venv_demo/bin/pip", "freeze"],
    capture_output=True, text=True
)
print(result.stdout)
# Note: requests pulled in certifi, charset-normalizer, idna, urllib3 as transitive deps.
# pip freeze captures all of them, but doesn't tell you WHICH ones you explicitly asked for.

**Key observation:** `pip freeze` is honest — it records everything — but it doesn't
separate *your* deps from *their* deps. That makes it hard to upgrade or audit later.
That's the gap `pip-tools` and `poetry` fill.

---
## Part 3 — `pip-tools`: compile a lockfile from first principles

Two tools, two jobs:
- `pip-compile` — reads `requirements.in` (your intent) → writes `requirements.txt` (the full resolved + hashed tree)
- `pip-sync`    — installs *exactly* that tree, removing anything extra

The separation matters: you edit `requirements.in`, machine generates `requirements.txt`.

In [ ]:
# Install pip-tools into the demo venv
result = subprocess.run(
    [f"{DEMO_DIR}/venv_demo/bin/pip", "install", "pip-tools", "--quiet"],
    capture_output=True, text=True
)
print(result.stderr[-500:] if result.returncode != 0 else "pip-tools installed")

In [ ]:
# Write a minimal requirements.in — only the packages WE care about, with loose constraints
req_in = """\
scikit-learn>=1.3
pandas
"""

with open(f"{DEMO_DIR}/requirements.in", "w") as f:
    f.write(req_in)

print("requirements.in:")
print(req_in)

In [ ]:
# pip-compile resolves the full dep tree and writes a hashed requirements.txt
# --generate-hashes adds SHA-256 for every package (supply-chain protection)
result = subprocess.run(
    [
        f"{DEMO_DIR}/venv_demo/bin/pip-compile",
        "--generate-hashes",
        "--output-file", f"{DEMO_DIR}/requirements.txt",
        f"{DEMO_DIR}/requirements.in",
    ],
    capture_output=True, text=True, cwd=DEMO_DIR
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-1000:])

In [ ]:
# Inspect the compiled lockfile — note how many more packages appeared
with open(f"{DEMO_DIR}/requirements.txt") as f:
    content = f.read()

print(content[:3000])  # first 3000 chars to keep output manageable

Each entry shows:
- exact pinned version (`==`)
- one or more `--hash=sha256:...` lines
- a comment showing which top-level dep pulled it in

If someone swaps a package on PyPI for a malicious one at the same version number, `pip install --require-hashes` will refuse it. `pip freeze` alone doesn't protect against this.

---
## Part 4 — `poetry`: all-in-one dep management

Poetry replaces `venv` + `pip` + `pip-tools` + `setuptools` with a single tool.

| Concept | pip-tools equivalent | poetry equivalent |
|---------|---------------------|-------------------|
| Declare direct deps | `requirements.in` | `[tool.poetry.dependencies]` in `pyproject.toml` |
| Lockfile | `requirements.txt` (compiled) | `poetry.lock` |
| Install locked deps | `pip-sync requirements.txt` | `poetry install` |
| Add a new dep | edit `.in`, re-run `pip-compile` | `poetry add <pkg>` |
| Manage the venv | manual | automatic |
| Build / publish | separate tools | `poetry build` / `poetry publish` |

In [ ]:
# Check if poetry is available
result = subprocess.run(["which", "poetry"], capture_output=True, text=True)
poetry_available = result.returncode == 0
print(f"poetry found: {poetry_available}")
if poetry_available:
    result2 = subprocess.run(["poetry", "--version"], capture_output=True, text=True)
    print(result2.stdout.strip())

In [ ]:
# If poetry isn't installed, we can install it via pip (fine for demo purposes)
# In production you'd install it via the official installer: https://install.python-poetry.org
if not poetry_available:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "poetry", "--quiet"],
        capture_output=True, text=True
    )
    print(result.stderr[-300:] if result.returncode != 0 else "poetry installed")
    # Re-check
    result2 = subprocess.run(["poetry", "--version"], capture_output=True, text=True)
    print(result2.stdout.strip())

In [ ]:
# Initialize a new project in the demo dir
POETRY_DIR = f"{DEMO_DIR}/poetry_demo"
os.makedirs(POETRY_DIR, exist_ok=True)

result = subprocess.run(
    ["poetry", "init", "--no-interaction",
     "--name", "demo",
     "--description", "",
     "--author", "Demo User <demo@example.com>",
     "--python", ">=3.9"],
    capture_output=True, text=True, cwd=POETRY_DIR
)
print(result.stdout or "pyproject.toml created")
if result.returncode != 0:
    print(result.stderr[-500:])

In [ ]:
# Inspect the generated pyproject.toml
with open(f"{POETRY_DIR}/pyproject.toml") as f:
    print(f.read())

In [ ]:
# Add a dependency — poetry resolves + updates poetry.lock automatically
# --no-interaction skips the venv creation prompt; POETRY_VIRTUALENVS_CREATE=false
# avoids creating a venv so this runs cleanly inside the notebook's own environment
env = os.environ.copy()
env["POETRY_VIRTUALENVS_CREATE"] = "false"

result = subprocess.run(
    ["poetry", "add", "requests", "--no-interaction"],
    capture_output=True, text=True, cwd=POETRY_DIR, env=env
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-600:])

In [ ]:
# After 'poetry add', pyproject.toml lists the direct dep with a constraint
with open(f"{POETRY_DIR}/pyproject.toml") as f:
    print(f.read())

In [ ]:
# poetry.lock pins the full resolved tree — similar to pip-tools' compiled requirements.txt
lock_path = f"{POETRY_DIR}/poetry.lock"
if os.path.exists(lock_path):
    with open(lock_path) as f:
        content = f.read()
    print(content[:2000])  # first entry or two
else:
    print("poetry.lock not found — 'poetry add' may have skipped lock creation without a venv")

---
## Part 5 — `pip-tools` vs `poetry`: side-by-side

Both produce a hashed lockfile. The difference is scope and opinion.

In [ ]:
comparison = """
┌─────────────────┬──────────────────────────────────┬──────────────────────────────────┐
│                 │           pip-tools              │             poetry               │
├─────────────────┼──────────────────────────────────┼──────────────────────────────────┤
│ Lockfile        │ requirements.txt (pip-compatible)│ poetry.lock (poetry-specific)    │
│ Venv management │ manual                           │ automatic                        │
│ Add a dep       │ edit .in, re-run pip-compile     │ poetry add <pkg>                 │
│ Build / publish │ separate tools (build, twine)    │ built-in (poetry build/publish)  │
│ CI portability  │ any pip understands the output   │ needs poetry installed in CI     │
│ Learning curve  │ low                              │ medium                           │
│ Best for        │ ML stacks, conda pipelines       │ pure-Python services, libraries  │
└─────────────────┴──────────────────────────────────┴──────────────────────────────────┘
"""
print(comparison)

---
## Part 6 — `conda`: when C/CUDA deps enter the picture

`conda` manages Python versions and binary (C/Fortran/CUDA) libraries alongside Python packages.  
This makes it the right tool when `pip` can't easily install something (e.g., GPU drivers, `faiss`, `lightgbm` with optimised BLAS).

An `environment.yml` is the conda equivalent of `requirements.in`:

In [ ]:
conda_yml = """\
name: ml-demo
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.11          # conda manages the Python version itself
  - numpy=1.26
  - scikit-learn>=1.3
  - pandas
  - pip                  # also include pip-installed packages below
  - pip:
    - mlflow>=2.0
"""
print(conda_yml)

In [ ]:
# Check if conda is available in this environment
result = subprocess.run(["conda", "--version"], capture_output=True, text=True)
conda_available = result.returncode == 0
print(f"conda found: {conda_available}")
if conda_available:
    print(result.stdout.strip())
    # List existing envs
    result2 = subprocess.run(["conda", "env", "list"], capture_output=True, text=True)
    print(result2.stdout)

In [ ]:
# Creating a full conda env from the yml above takes several minutes — we skip it here.
# The commands you'd run in a terminal:
conda_cmds = """\
# Create from file
conda env create -f environment.yml

# Activate
conda activate ml-demo

# Export an exact snapshot (including build strings — fully reproducible)
conda env export > environment_locked.yml

# Export only the packages you explicitly chose (easier to maintain across OS)
conda env export --from-history > environment.yml
"""
print(conda_cmds)

---
## Part 7 — Docker: freeze the whole OS

All the tools above still leave the OS and system libraries variable.  
Docker packages everything into an image that runs identically anywhere.

A minimal ML `Dockerfile` that layers the tools we've discussed:

In [ ]:
dockerfile = """\
# ── base image pins the OS + Python version ───────────────────────────────────
FROM python:3.11-slim

WORKDIR /app

# ── system deps (if any C libs needed, apt-get install here) ─────────────────
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# ── install pip-tools ────────────────────────────────────────────────────────
RUN pip install --no-cache-dir pip-tools

# ── copy only the lockfile first (layer caching: deps rebuilt only if lock changes)
COPY requirements.txt .
RUN pip install --no-cache-dir --require-hashes -r requirements.txt

# ── copy source code ─────────────────────────────────────────────────────────
COPY src/ ./src/

# ── default command ──────────────────────────────────────────────────────────
CMD ["python", "-m", "src.pipeline.train"]
"""
print(dockerfile)

In [ ]:
# Check if docker is available
result = subprocess.run(["docker", "--version"], capture_output=True, text=True)
docker_available = result.returncode == 0
print(f"docker found: {docker_available}")
if docker_available:
    print(result.stdout.strip())

In [ ]:
# Key docker commands for an ML project:
docker_cmds = """\
# Build the image (tags it 'my-ml-project:latest')
docker build -t my-ml-project .

# Run training — container is deleted after exit (--rm)
docker run --rm my-ml-project python -m src.pipeline.train

# Mount a local data dir so the container can read it without baking data into the image
docker run --rm -v $(pwd)/data:/app/data my-ml-project python -m src.pipeline.train

# Start an interactive shell for debugging
docker run --rm -it my-ml-project /bin/bash

# Inspect what's inside the image
docker run --rm my-ml-project pip list
"""
print(docker_cmds)

---
## Part 8 — Decision guide

Run the cell below to print a quick cheat-sheet based on your situation.

In [ ]:
guide = """\
Situation                                          → Recommended tool(s)
──────────────────────────────────────────────────────────────────────────────
Quick one-off script, no teammates                 → venv + pip freeze
Pure-Python project, team of 2-5, no PyPI publish  → venv + pip-tools
Pure-Python library / service you'll publish       → poetry
Data science, need specific Python / CUDA / BLAS   → conda
Hand off to CI / another machine / prod container  → + Docker (on top of any above)
ML training pipeline in production                 → conda + Docker  (most common)
──────────────────────────────────────────────────────────────────────────────

Rule of thumb: add one layer at a time.
  Start with venv. Add pip-tools when a teammate says 
.
  Add Docker when you hit 
 with a CI system or colleague.
  Switch to conda when pip can't resolve a binary dep.
"""
print(guide)

In [ ]:
# Cleanup: remove the demo directory
shutil.rmtree(DEMO_DIR, ignore_errors=True)
print("Demo directory removed.")